# scPLAD end-to-end reproduction workflow

Reproduce and audit the manuscript figure from archived results. Model training is documented but is **not executed**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
if not (ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv").exists():
    raise FileNotFoundError("Run this notebook from the archive root or notebooks directory.")

REGISTRY = pd.read_csv(
    ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv",
    sep="\t",
    keep_default_na=False,
)

default_config = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
config_path = Path(os.environ.get("SCPLAD_REPRO_CONFIG", default_config)).expanduser().resolve()
REPRO_CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
config_dir = config_path.parent

def resolve_config_path(key, fallback):
    env_key = {
        "source_data_root": "SCPLAD_SOURCE_DATA_ROOT",
        "reproduced_root": "SCPLAD_REPRODUCED_ROOT",
    }[key]
    value = os.environ.get(env_key, REPRO_CONFIG.get("paths", {}).get(key, fallback))
    path = Path(value).expanduser()
    return path if path.is_absolute() else (config_dir / path).resolve()

SOURCE_DATA = resolve_config_path("source_data_root", str(ARCHIVE / "source_data"))
REPRO = resolve_config_path("reproduced_root", str(ARCHIVE / "reproduced"))
REPRO.mkdir(parents=True, exist_ok=True)
os.environ["SCPLAD_REPRO_CONFIG"] = str(config_path)
os.environ["SCPLAD_SOURCE_DATA_ROOT"] = str(SOURCE_DATA)
os.environ["SCPLAD_REPRODUCED_ROOT"] = str(REPRO)
print(f"Figure archive: {ARCHIVE}")
print(f"Input mode: {REPRO_CONFIG['mode']}")
print(f"Source data: {SOURCE_DATA}")
print(f"Output root: {REPRO}")

## Choose a reproduction route

- **Provided results**: validate and render all figures from the
  manuscript source tables. This route does not require a GPU.
- **Custom data**: prepare data, train PatchAE and scPLAD, generate
  cells, evaluate them, and render figures. Copy the custom template
  first and review every path and command.

Training is never started automatically from this notebook.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
PROVIDED_CONFIG = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
CUSTOM_K562 = ARCHIVE / "reproducibility" / "configs" / "custom_data.template.json"
CUSTOM_CROSS_CELL = ARCHIVE / "reproducibility" / "configs" / "custom_cross_cell_line.template.json"

# Change this path to a completed copy of the matching custom template.
CONFIG = PROVIDED_CONFIG
config = json.loads(CONFIG.read_text(encoding="utf-8"))
print("Selected mode:", config["mode"])
print("Selected task:", config["task"])
print("K562 template:", CUSTOM_K562)
print("Cross-cell-line template:", CUSTOM_CROSS_CELL)
display(config)

## Validate the selected configuration

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(ARCHIVE / "reproducibility" / "scplad_repro.py"),
        "--config", str(CONFIG),
        "--validate-only",
    ],
    check=True,
)

## Inspect the complete custom-data command chain

This dry run prints data preparation, PatchAE training, scPLAD
training, inference, evaluation, and figure commands without
executing them. It is safe to run on a laptop.

In [ ]:
if config["mode"] == "custom":
    subprocess.run(
        [
            sys.executable,
            str(ARCHIVE / "reproducibility" / "scplad_repro.py"),
            "--config", str(CONFIG),
            "--stages", "all",
            "--dry-run",
        ],
        check=True,
    )
else:
    print("Provided-results mode uses archived outputs and skips training.")

## Execute

To reproduce manuscript panels from provided results, run the next
cell. For custom data, execute training from a terminal only after
reviewing the dry run and adding `--allow-training`.

In [ ]:
if config["mode"] == "provided":
    subprocess.run(
        [
            sys.executable,
            str(ARCHIVE / "reproducibility" / "scplad_repro.py"),
            "--config", str(CONFIG),
            "--stages", "figures",
        ],
        check=True,
    )
else:
    print(
        "Custom mode is configured. Run the reviewed stages from a "
        "terminal; training remains protected by --allow-training."
    )